# 04 - Final comparison (baselines vs proposed model)

The final comparison between the proposed machine-learning model and the baseline models.

## 1. Imports and configuration

In [ ]:
import json
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

pd.set_option("display.max_columns", 200)

PROJECT_ROOT  = Path("..")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FIG_DIR = ARTIFACTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

BASELINES_JSON = ARTIFACTS_DIR / "baselines_detailed.json"
FINAL_META_GLOB = ARTIFACTS_DIR / "final_model" / "final_model_meta.json"

TEST_DAYS = ["2025-04-11", "2025-04-20", "2025-10-07"]

In [ ]:
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

MODEL_LABELS = {
    "A_Physical_Simulation": "Baseline A — normalized physical simulation",
    "B_ClearSky": "Baseline B — clear-sky (GHI)",
    "C_XGBoost_Meteo": "Baseline C — meteo only (XGBoost)",
    "Final (sim+ML)": "Final model (sim + ML)",
    "Final_sim_ML": "Final model (sim + ML)",
}

COL_LABELS = {
    "global_r2": "Global R²",
    "global_rmse": "Global RMSE (W/m²)",
    "global_mae": "Global MAE (W/m²)",
    "global_mbe": "Global MBE (W/m²)",
    "n_train": "Train samples",
    "n_test": "Test samples",
    "model": "Model",
}

def pretty_model(name: str) -> str:
    return MODEL_LABELS.get(name, name)

## 2. Loading and normalization utilities

In [ ]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def meta_to_flat_row(meta: dict, model_name: str = "Final (sim+ML)"):
    """
    Convierte model_final_meta.json a una fila compatible con baselines_results.csv
    """
    g = meta.get("metrics_global", {})
    row = {
        "model": model_name,
        "n_train": meta.get("n_train", np.nan),
        "n_test": meta.get("n_test", np.nan),
        "global_r2": g.get("r2", np.nan),
        "global_mae": g.get("mae", np.nan),
        "global_rmse": g.get("rmse", np.nan),
        "global_mbe": g.get("mbe", np.nan),
    }

    for drow in meta.get("metrics_by_day", []):
        d = drow.get("date")
        if not d:
            continue
        row[f"{d}_rmse"] = drow.get("rmse", np.nan)
        row[f"{d}_mae"]  = drow.get("mae", np.nan)
        row[f"{d}_r2"]   = drow.get("r2", np.nan)
        row[f"{d}_mbe"]  = drow.get("mbe", np.nan)

    return row

In [ ]:
def styled_metrics_table(df: pd.DataFrame, caption: str):
    return (
        df.style
        .format({
            "global_r2": "{:.3f}",
            "global_rmse": "{:.1f}",
            "global_mae": "{:.1f}",
            "global_mbe": "{:.1f}",
        }, na_rep="—")
        .set_caption(caption)
    )

## 3. Loading the results

In [ ]:
# --- Baselines (A,B,C) ---
baselines = load_json(BASELINES_JSON)
rows_base = [meta_to_flat_row(b, model_name=b.get("model","Baseline")) for b in baselines]
df_base = pd.DataFrame(rows_base)

# --- Final model meta (latest iteration meta json) ---
candidates = sorted(FINAL_META_GLOB.parent.glob(FINAL_META_GLOB.name))
if len(candidates) == 0:
    raise FileNotFoundError(f"No meta found in: {FINAL_META_GLOB}")
FINAL_META = max(candidates, key=lambda p: p.stat().st_mtime)
meta_final = load_json(FINAL_META)
row_final = meta_to_flat_row(meta_final, model_name="Final (sim+ML)")

df_all = pd.concat([df_base, pd.DataFrame([row_final])], ignore_index=True)
df_all = df_all.sort_values("global_rmse", ascending=True).reset_index(drop=True)

print(f"[OK] Baselines loaded from: {BASELINES_JSON}")
print(f"[OK] Final meta loaded from: {FINAL_META}")
df_all

## 4. Global comparison table

In [ ]:
tbl_global = pd.DataFrame({
    "Model": df_all["model"].apply(pretty_model),
    "Global R²": df_all["global_r2"],
    "Global RMSE (W/m²)": df_all["global_rmse"],
    "Global MAE (W/m²)": df_all["global_mae"],
    "Global MBE (W/m²)": df_all["global_mbe"],
    "Train samples": df_all["n_train"],
    "Test samples": df_all["n_test"],
})

tbl_global = tbl_global.sort_values("Global RMSE (W/m²)", ascending=True)

display(
    tbl_global.style
    .format({
        "Global R²": "{:.3f}",
        "Global RMSE (W/m²)": "{:.1f}",
        "Global MAE (W/m²)": "{:.1f}",
        "Global MBE (W/m²)": "{:.1f}",
        "Train samples": "{:.0f}",
        "Test samples": "{:.0f}",
    }, na_rep="—")
    .set_caption("Global model comparison")
)


## 5. Plots globales (RMSE y R²)

In [ ]:
df_plot = df_all.copy()
df_plot["model_pretty"] = df_plot["model"].map(pretty_model)

plt.figure(figsize=(9, 4))
plt.bar(df_plot["model_pretty"], df_plot["global_rmse"])
plt.ylabel("Global RMSE (W/m²)")
plt.title("Global comparison — RMSE")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(df_plot["model_pretty"], df_plot["global_r2"])
plt.ylabel("Global R²")
plt.title("Global comparison — R²")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()



## 6. Per-day comparison

In [ ]:
def by_day_table(df: pd.DataFrame, day: str) -> pd.DataFrame:
    cols = ["model", f"{day}_r2", f"{day}_rmse", f"{day}_mae", f"{day}_mbe"]
    out = df[cols].copy()
    out = out.sort_values(f"{day}_rmse", ascending=True).reset_index(drop=True)
    return out

for day in TEST_DAYS:
    df_day = by_day_table(df_all, day)

    df_day["model_label"] = df_day["model"].apply(pretty_model)

    # ---------- Table ----------
    display(
        df_day[["model_label", f"{day}_r2", f"{day}_rmse", f"{day}_mae", f"{day}_mbe"]]
        .style
        .format({
            f"{day}_r2": "{:.3f}",
            f"{day}_rmse": "{:.1f}",
            f"{day}_mae": "{:.1f}",
            f"{day}_mbe": "{:.1f}",
        }, na_rep="—")
        .set_caption(f"Per-day comparison — {day} (sorted by RMSE)")
    )

    # ---------- Bar plot ----------
    plt.figure(figsize=(9, 4))
    plt.bar(df_day["model_label"], df_day[f"{day}_rmse"])
    plt.ylabel("RMSE (W/m²)")
    plt.title(f"Per-day comparison — RMSE ({day})")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


## 7. Series temporales comparativas

In [ ]:
DAYS_TO_PLOT = ["2025-10-07", "2025-04-11", "2025-04-20"]
SENSOR_NAME = "P1"

MODELS = [
    "A_Physical_Simulation",
    "B_ClearSky",
    "C_XGBoost_Meteo",
]

FINAL_MODEL_LABEL = "Final_sim_ML"
FINAL_PREDS_PATTERN = "preds_{}_{}.csv" 

PROJECT_ROOT = Path("..")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "predictions"


def _sanitize_label(s: str) -> str:
    return "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in s)


def load_preds_csv(path: Path) -> pd.DataFrame:
    """
    Load a predictions CSV and rebuild a time index if one is present.
    Supports: timestamp, utc, datetime, time, Unnamed: 0
    """
    if not path.exists():
        raise FileNotFoundError(f"Not found: {path}")

    df = pd.read_csv(path)

    time_candidates = ["timestamp", "utc", "datetime", "time", "Unnamed: 0"]
    time_col = next((c for c in time_candidates if c in df.columns), None)

    if time_col is not None:
        ts = pd.to_datetime(df[time_col], errors="coerce", utc=True)
        if ts.notna().sum() > 0:
            df = df.drop(columns=[time_col])
            df.index = ts
            df = df.sort_index()

    return df


def filter_sensor(df: pd.DataFrame, sensor_name: str) -> pd.DataFrame:
    """
    Filter by sensor:
    - sensor_name (new export from the final model)
    - sensor (if present)
    - dummies sensor_P1, etc.
    """
    if "sensor_name" in df.columns:
        return df[df["sensor_name"].astype(str) == str(sensor_name)].copy()

    if "sensor" in df.columns:
        return df[df["sensor"].astype(str) == str(sensor_name)].copy()

    col = f"sensor_{sensor_name}"
    if col in df.columns:
        return df[df[col] == 1].copy()

    return df.copy()


def _dedup_time_index_for_plot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Avoid the 'comb' effect when plotting:
    - sort by time index
    - if timestamps are duplicated, aggregate by mean
    """
    if isinstance(df.index, pd.DatetimeIndex):
        df = df.sort_index()
        if df.index.has_duplicates:
            # Timeindex duplicates: mean per timestamp (for plotting)
            df = df.groupby(df.index).mean(numeric_only=True)
    return df

BASELINE_STYLES = {
    "A_Physical_Simulation": dict(linestyle="--", linewidth=1.5),
    "B_ClearSky": dict(linestyle="--", linewidth=1.5),
    "C_XGBoost_Meteo": dict(linestyle="--", linewidth=1.5),
}

FINAL_STYLE = dict(color="red", linestyle="-", linewidth=2.0)
REAL_STYLE  = dict(color="black", linestyle="-", linewidth=1.8)

def plot_day_comparison(day: str, sensor_name: str, models: list):
    plt.figure(figsize=(14, 4))

    any_loaded = False
    real_plotted = False

    # Baselines
    for m in models:
        p = ARTIFACTS_DIR / f"preds_{m}_{day}.csv"
        if not p.exists():
            print(f"[WARN] Not found: {p.name}")
            continue

        df = load_preds_csv(p)
        df = filter_sensor(df, sensor_name)
        df = _dedup_time_index_for_plot(df)
        
        if isinstance(df.index, pd.DatetimeIndex):
            day_start = pd.Timestamp(day).tz_localize("UTC")
            day_end = day_start + pd.Timedelta(days=1)
            df = df.loc[(df.index >= day_start) & (df.index < day_end)]

        if df.empty:
            print(f"[WARN] Empty after filtering sensor {sensor_name}: {p.name}")
            continue

        if (not real_plotted) and ("real_irradiance" in df.columns):
            plt.plot(df.index, df["real_irradiance"].values, label="Real", **REAL_STYLE)
            real_plotted = True

        if "pred" not in df.columns:
            print(f"[WARN] No 'pred' column in {p.name}")
            continue
        style = BASELINE_STYLES.get(m, dict(linestyle="--", linewidth=1.5))
        plt.plot(df.index, df["pred"].values, label=pretty_model(m), **style)
        any_loaded = True

    # Final model
    final_name = FINAL_PREDS_PATTERN.format(_sanitize_label(FINAL_MODEL_LABEL), day)
    p_final = ARTIFACTS_DIR / final_name

    if p_final.exists():
        dfF = load_preds_csv(p_final)
        dfF = filter_sensor(dfF, sensor_name)
        dfF = _dedup_time_index_for_plot(dfF)
        
        if isinstance(dfF.index, pd.DatetimeIndex):
            day_start = pd.Timestamp(day).tz_localize("UTC")
            day_end = day_start + pd.Timedelta(days=1)
            dfF = dfF.loc[(dfF.index >= day_start) & (dfF.index < day_end)]

        if not dfF.empty and "pred" in dfF.columns:
            if (not real_plotted) and ("real_irradiance" in dfF.columns):
                plt.plot(dfF.index, dfF["real_irradiance"].values, label="Real", **REAL_STYLE)
                real_plotted = True

            plt.plot(dfF.index, dfF["pred"].values, label=pretty_model(FINAL_MODEL_LABEL), **FINAL_STYLE)
            any_loaded = True
    else:
        print(f"[INFO] Final-model preds not found: {p_final.name}")

    if not any_loaded:
        print(f"[WARN] Could not load any CSV for {day} / {sensor_name}.")
        plt.close()
        return

    plt.title(f"Irradiance time series — {day} — sensor {sensor_name}")

    # X label
    ax = plt.gca()
    if any(isinstance(line.get_xdata(), (np.ndarray, list)) for line in ax.lines) and isinstance(df.index, pd.DatetimeIndex):
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))   
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        plt.xlabel("Time (local)")
    else:
        plt.xlabel("Samples")

    plt.ylabel("Irradiance [W/m²]")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


for d in DAYS_TO_PLOT:
    plot_day_comparison(d, SENSOR_NAME, MODELS)